# TPOSE24 validation vs OSCAR (surface currents)

> **NOTE — not executed.** The OSCAR archive on this system (`/data/SO3/edavenport/OSCAR_2012/`) only contains pentads for **March–May 2012**, which do **not** overlap the Oct–Dec 2012 model run.  This notebook is written and ready: once OSCAR files covering the run period are downloaded into `OSCAR_DIR`, it will execute as-is.  The loader and the guard below handle the missing-data case gracefully.

OSCAR pentad (5-day) surface currents (`u`, `v`, ~15 m) are compared to the model surface velocity (`UVEL`, `VVEL`, top level ≈ 0.5 m — note the small depth mismatch).  The model is daily-averaged then averaged over each OSCAR pentad window and regridded onto the OSCAR grid.

**1-D**: equatorial zonal surface current vs longitude (the SEC / cold-tongue surface signature); surface-speed RMSE time series; meridional profile of zonal current at the central longitude.

**2-D**: maps of time-mean zonal current, of the U bias (model − OSCAR), and of the V bias for each run.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  MODEL RUNS TO VALIDATE
#  Each entry is (label_for_figures, run_directory).
#  Add or remove entries here — every figure below updates its number
#  of lines / panels automatically.
# ════════════════════════════════════════════════════════════════════
MODELS = [
    ('Ri7', '/data/SO3/edavenport/tpose24/oct2012_3month_transp_cons'),
    ('Ri3', '/data/SO3/edavenport/tpose24/oct2012_TP6Vel_3month_Ri3'),
    ('Ri5', '/data/SO3/edavenport/tpose24/oct2012_TP6Vel_3month_Ri5'),
]

OUTDIR    = 'OSCAR_comparison'
OSCAR_DIR = '/data/SO3/edavenport/OSCAR_2012'   # pentad files (gzip-compressed netCDF)


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cmocean.cm as cmo

import obs_validation_utils as u

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11})
os.makedirs(OUTDIR, exist_ok=True)
print(f'{len(MODELS)} model run(s):', [m[0] for m in MODELS])
print('Figures ->', OUTDIR)


In [ ]:
# ── OSCAR loader ─────────────────────────────────────────────────────
# OSCAR files are gzip-compressed netCDF (their on-disk name ends in .nc
# but `file` reports gzip).  We transparently decompress each into a
# temp file, stamp it with its pentad date, and concatenate.
import glob, gzip, shutil, tempfile

def _maybe_gunzip(f, tmpdir):
    """Return a path to a plain netCDF file.  Some OSCAR files in the
    archive are gzip-compressed (despite the .nc name) and some are not,
    so we sniff the magic bytes and only decompress when needed."""
    with open(f, 'rb') as fh:
        magic = fh.read(2)
    if magic == b'\x1f\x8b':            # gzip
        dec = os.path.join(tmpdir, os.path.basename(f))
        with gzip.open(f, 'rb') as fi, open(dec, 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        return dec
    return f


def load_oscar(oscar_dir, t0, t1):
    """Load OSCAR pentads overlapping [t0, t1] -> Dataset(u, v) on a
    daily-comparable 'time' axis.  Returns None if nothing overlaps."""
    files = sorted(glob.glob(os.path.join(oscar_dir, 'oscar_*.nc')))
    if not files:
        return None
    tmpdir = tempfile.mkdtemp(prefix='oscar_')
    frames = []
    for f in files:
        dec = _maybe_gunzip(f, tmpdir)
        # time units are the non-standard 'day since 1992-10-05'; read the
        # raw integer (decode_times=False) and build the date ourselves.
        d = xr.open_dataset(dec, decode_times=False)
        t = pd.Timestamp('1992-10-05') + pd.Timedelta(days=int(d.time.values[0]))
        d = d.isel(time=0, depth=0, year=0, drop=True)
        d = d.assign_coords(time=t).expand_dims('time')
        frames.append(d[['u', 'v']])
    ds = xr.concat(frames, dim='time').sortby('time')
    ds = u.to_0360(ds)
    sel = ds.sel(time=slice(t0, t1))
    return sel if sel.time.size else None


In [ ]:
# ── Load every model surface velocity; match to OSCAR pentads ────────
# UVEL lives on (XG, YC), VVEL on (XC, YG); each is regridded from its
# own staggered coordinate onto the OSCAR centre grid.
results = {}
obs_u = obs_v = None
have_oscar = True

for i, (label, run_dir) in enumerate(MODELS):
    print(f'[{label}] loading {run_dir}')
    ds   = u.load_tpose24(run_dir, prefix=['diag_state'])
    uvel = ds.UVEL.isel(Z=0).where(ds.UVEL.isel(Z=0) != 0)
    vvel = ds.VVEL.isel(Z=0).where(ds.VVEL.isel(Z=0) != 0)
    ud   = u.daily_mean(uvel)
    vd   = u.daily_mean(vvel)

    if obs_u is None:
        dom = u.model_domain(ds)
        t0  = str(ud.time.values[0])[:10]
        t1  = str(ud.time.values[-1])[:10]
        oscar = load_oscar(OSCAR_DIR, t0, t1)
        if oscar is None:
            have_oscar = False
            print('   No OSCAR pentads overlap the run period '
                  f'({t0} – {t1}); skipping execution.')
            break
        oscar = u.subset_domain(oscar, *dom).compute()
        obs_lon = oscar.longitude; obs_lat = oscar.latitude

    # average model daily fields over each OSCAR pentad [t, t+5d)
    def pentad_mean(daily, otime):
        out = []
        for t in otime.values:
            win = daily.sel(time=slice(pd.Timestamp(t),
                                       pd.Timestamp(t) + pd.Timedelta(days=4)))
            out.append(win.mean('time'))
        return xr.concat(out, dim='time').assign_coords(time=otime.values)

    um = u.regrid_model_to_obs(pentad_mean(ud, oscar.time), obs_lon, obs_lat,
                               x='XG', y='YC').compute()
    vm = u.regrid_model_to_obs(pentad_mean(vd, oscar.time), obs_lon, obs_lat,
                               x='XC', y='YG').compute()

    if obs_u is None:
        obs_u = oscar.u; obs_v = oscar.v
        lat = obs_u.latitude.values; lon = obs_u.longitude.values
        time = obs_u.time.values

    uo = obs_u.values; vo = obs_v.values
    results[label] = dict(
        u_mean  = np.nanmean(um.values, axis=0),
        v_mean  = np.nanmean(vm.values, axis=0),
        u_bias  = np.nanmean(um.values - uo, axis=0),
        v_bias  = np.nanmean(vm.values - vo, axis=0),
        spd_rmse_t = u.weighted_spatial_rmse(
            np.hypot(um.values, vm.values), np.hypot(uo, vo), lat),
    )

if have_oscar:
    obs_u_mean = np.nanmean(obs_u.values, axis=0)
    print('Aligned OSCAR pentads:', len(time))


In [ ]:
# ── 1-D: equatorial zonal surface current vs longitude (|lat|<1°) ────
assert have_oscar, 'No overlapping OSCAR data — see note at top of notebook.'

band = np.abs(lat) <= 1.0
w = np.cos(np.deg2rad(lat[band]))[:, None]
def band_mean(a2d):
    a = a2d[band, :]
    return np.nansum(a * w, axis=0) / np.nansum(w * np.isfinite(a), axis=0)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(lon, band_mean(obs_u_mean), color='k', lw=2.5, label='OSCAR')
for i, (label, _) in enumerate(MODELS):
    ax.plot(lon, band_mean(results[label]['u_mean']),
            color=u.model_color(i), lw=1.8, label=label)
ax.axhline(0, color='gray', lw=0.7, ls=':')
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Zonal current U (m s⁻¹)')
ax.set_title('Equatorial surface zonal current, 1°S–1°N')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(f'{OUTDIR}/oscar_1d_equatorial_U.png', dpi=150,
            bbox_inches='tight')
plt.show()


In [ ]:
# ── 1-D: surface-speed RMSE time series + meridional U profile ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
tt = pd.to_datetime(time)
for i, (label, _) in enumerate(MODELS):
    axes[0].plot(tt, results[label]['spd_rmse_t'], color=u.model_color(i),
                 lw=1.8, marker='o', ms=3, label=label)
axes[0].set_title('Surface-speed RMSE vs OSCAR')
axes[0].set_ylabel('RMSE (m s⁻¹)'); axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

ic = len(lon) // 2
axes[1].plot(np.nanmean(obs_u.values, axis=0)[:, ic], lat, color='k',
             lw=2.5, label='OSCAR')
for i, (label, _) in enumerate(MODELS):
    axes[1].plot(results[label]['u_mean'][:, ic], lat,
                 color=u.model_color(i), lw=1.8, label=label)
axes[1].axvline(0, color='gray', lw=0.7, ls=':')
axes[1].set_title(f'Zonal current U at {lon[ic]:.1f}°E')
axes[1].set_xlabel('U (m s⁻¹)'); axes[1].set_ylabel('Latitude (°N)')
axes[1].grid(alpha=0.3); axes[1].legend()
fig.tight_layout()
fig.savefig(f'{OUTDIR}/oscar_1d_speed_rmse_profile.png', dpi=150,
            bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-D: time-mean zonal current U (OSCAR + each model) ──────────────
n = len(MODELS) + 1
allv = [obs_u_mean] + [results[l]['u_mean'] for l, _ in MODELS]
vmax = np.nanmax([np.nanpercentile(np.abs(a), 99) for a in allv])
fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), sharex=True,
                         sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
im = axes[0].pcolormesh(lon, lat, obs_u_mean, cmap=cmo.balance,
                        vmin=-vmax, vmax=vmax, shading='auto')
axes[0].set_title('OSCAR')
for i, (label, _) in enumerate(MODELS):
    axes[i + 1].pcolormesh(lon, lat, results[label]['u_mean'],
                           cmap=cmo.balance, vmin=-vmax, vmax=vmax,
                           shading='auto')
    axes[i + 1].set_title(label)
for ax in axes:
    ax.axhline(0, color='gray', lw=0.5, ls=':'); ax.set_xlabel('Lon (°E)')
axes[0].set_ylabel('Lat (°N)')
fig.colorbar(im, ax=axes, label='Mean U (m s⁻¹)', shrink=0.85)
fig.suptitle('Time-mean zonal surface current', y=1.04)
fig.savefig(f'{OUTDIR}/oscar_2d_mean_U.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-D: U and V bias maps (model − OSCAR) ───────────────────────────
for comp, key, title in [('U', 'u_bias', 'Zonal'),
                         ('V', 'v_bias', 'Meridional')]:
    n = len(MODELS)
    vmax = np.nanmax([np.nanpercentile(np.abs(results[l][key]), 99)
                      for l, _ in MODELS])
    fig, axes = plt.subplots(1, n, figsize=(4.6 * n, 4), sharex=True,
                             sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for i, (label, _) in enumerate(MODELS):
        im = axes[i].pcolormesh(lon, lat, results[label][key],
                                cmap=cmo.balance, vmin=-vmax, vmax=vmax,
                                shading='auto')
        axes[i].set_title(f'{label} − OSCAR')
        axes[i].axhline(0, color='gray', lw=0.5, ls=':')
        axes[i].set_xlabel('Lon (°E)')
    axes[0].set_ylabel('Lat (°N)')
    fig.colorbar(im, ax=axes, label=f'{comp} bias (m s⁻¹)', shrink=0.85)
    fig.suptitle(f'{title} surface current bias (model − OSCAR)', y=1.04)
    fig.savefig(f'{OUTDIR}/oscar_2d_{comp.lower()}_bias_maps.png', dpi=150,
                bbox_inches='tight')
    plt.show()
